# RAFOS Sonification

In [1]:
import pandas as pd
import numpy as np
from scipy.io.wavfile import write

Let's define functions to create OpenSpace assets from data in this format:

In [2]:
def create_openspace_asset(df, cruise_id, vessel_name="Custom Vessel"):
    # 1. Determine time bounds from your data
    # We'll use these for the TimeFrameIntervals
    start_time = pd.to_datetime(df['unixTime'].min(), unit='s').strftime('%Y-%m-%dT%H:%M:%S.00Z')
    end_time = pd.to_datetime(df['unixTime'].max(), unit='s').strftime('%Y-%m-%dT%H:%M:%S.00Z')

    # 2. Define the Lua template
    # Note: We use double curly braces {{ }} for literal Lua tables
    # and single { } for python f-string variables.
    asset_template = f"""
local sun = asset.require("scene/solarsystem/sun/transforms")
local earthTransforms = asset.require("scene/solarsystem/planets/earth/earth")

-- Define the float model resource
local floatModel = asset.resource({{
    Name = "{cruise_id} Model",
    Type = "UrlSynchronization",
    Identifier = "{cruise_id}_model_res",
    -- Url = "https://github.com/CreativeTools/3DBenchy/raw/master/Single-part/3DBenchy.stl",
    Url = "https://raw.githubusercontent.com/kcollins/openspace_rafos/master/oceanographic_argo_profiling_float.glb",
    Version = 1
}})

-- The keyframes for the float's trajectory (generated by openspace_rvdata)
local floatKeyframes = asset.require("./{cruise_id}_keyframes.asset")

-- Define the float's position
local floatPosition = {{
    Identifier = "FloatPosition_{cruise_id}",
    Parent = earthTransforms.Earth.Identifier,
    TimeFrame = {{
        Type = "TimeFrameInterval",
        Start = "{start_time}",
        End = "{end_time}"
    }},
    Transform = {{
        Translation = {{
            Type = "TimelineTranslation",
            Keyframes = floatKeyframes.keyframes
        }}
    }},
    GUI = {{
        Name = "{cruise_id} Position",
        Path = "/Float Tracks/{vessel_name}/{cruise_id}"
    }}
}}

-- Define the float model to be rendered
local floatRenderable = {{
    Identifier = "FloatModel_{cruise_id}",
    Parent = floatPosition.Identifier,
    TimeFrame = {{
        Type = "TimeFrameInterval",
        Start = "{start_time}",
        End = "{end_time}"
    }},
    Transform = {{
        Scale = {{
            Type = "StaticScale",
            Scale = 1000.0
        }}
    }},
    Renderable = {{
        Type = "RenderableModel",
        GeometryFile = floatModel .. "oceanographic_argo_profiling_float.glb",
        LightSources = {{
            sun.LightSource,
            {{
                Identifier = "Camera",
                Type = "CameraLightSource",
                Intensity = 0.5
            }}
        }}
    }},
    GUI = {{
        Name = "{vessel_name} Model",
        Path = "/Float Tracks/{vessel_name}"
    }}
}}

-- Define the trail
local floatTrail = {{
    Identifier = "FloatTrail_{cruise_id}",
    Parent = earthTransforms.Earth.Identifier,
    Renderable = {{
        Type = "RenderableTrailTrajectory",
        Enabled = true,
        Translation = {{
            Type = "TimelineTranslation",
            Keyframes = floatKeyframes.keyframes
        }},
        Color = {{ 1.0, 0.5, 0.0 }},
        StartTime = "{start_time}",
        EndTime = "{end_time}",
        SampleInterval = 60,
        EnableFade = true
    }},
    GUI = {{
        Name = "{cruise_id} Trail",
        Path = "/Float Tracks/{vessel_name}/{cruise_id}",
        Focusable = false
    }}
}}

asset.onInitialize(function()
    openspace.addSceneGraphNode(floatPosition)
    openspace.addSceneGraphNode(floatRenderable)
    openspace.addSceneGraphNode(floatTrail)
end)

asset.onDeinitialize(function()
    openspace.removeSceneGraphNode(floatTrail)
    openspace.removeSceneGraphNode(floatRenderable)
    openspace.removeSceneGraphNode(floatPosition)
end)

asset.export(floatPosition)
asset.export(floatRenderable)
asset.export(floatTrail)
"""

    with open(f"{cruise_id}.asset", "w") as f:
        f.write(asset_template.strip())

    print(f"Asset created! Ensure {cruise_id}_keyframes.asset exists in the same folder.")



In [3]:
import pandas as pd

def create_openspace_keyframes(df, buoy, cruise):
    """
    Generates an OpenSpace keyframe asset file (.txt) from a pandas DataFrame.
    """
    # Ensure the datetime column is in actual datetime objects
    # Using the 'datetime' column provided in your example
    df['datetime'] = pd.to_datetime(df['datetime'])

    lines = []
    lines.append("local keyframes = {")

    for _, row in df.iterrows():
        # OpenSpace expects ISO format: YYYY-MM-DDTHH:MM:SS
        iso_time = row['datetime'].strftime('%Y-%m-%dT%H:%M:%S')

        entry = f'''  ["{iso_time}"] = {{
    Type = "GlobeTranslation",
    Globe = "Earth",
    Longitude = {row['Longitude (W)']},
    Latitude = {row['Latitude (N)']},
    Altitude = 100,
    Temperature = {row['Temperature (C)']},
    Pressure = {row['Pressure (dbar)']},
    UseHeightmap = true
  }},'''
        lines.append(entry)

    lines.append("}\n")
    lines.append('asset.export("keyframes", keyframes)\n')

    # Metadata section
    meta = f'''asset.meta = {{
  Name = "Float Track Position: {buoy}",
  Description = [[This asset provides position information for the float track for the cruise {cruise}: Processed Trackline Navigation Data: One Minute Resolution]],
  Author = "OpenSpace Team",
  URL = "https://www.openspaceproject.com",
  License = "MIT license"
}}'''
    lines.append(meta)

    # Save to file
    filename = f"{buoy}_keyframes.asset"
    with open(filename, "w") as f:
        f.write("\n".join(lines))

    print(f"Asset file created: {filename}")

# --- Example Usage ---
# df = pd.read_csv("your_data.csv")
# create_openspace_keyframes(df, "Buoy123", "NorthAtlantic2026")

...And apply those functions to each of the datasets we have:

In [4]:
for buoy in ["rafos1060", "rafos1134", "rafos1232", "rafos1455", "rafos1496"]:
    # (NB: I know they're floats, not buoys -- but "float" is a reserved word!)
    filename = "Data/" + buoy+".csv"
    df = pd.read_csv(filename)
    df = df.dropna() # drop any NaN rows
    # Convert to datetime
    df['datetime'] = pd.to_datetime(df['Date (yyyy-MMM-dd hh:mm)'], format='mixed', utc=True)
    df['unixTime'] = df['datetime'].astype('int64') // 10**6
    # To verify, print the first value:
    # print(f"Verified Unix Time: {df['unixTime'].iloc[0]}")

    # Create OpenSpace asset files:
    print(f"Creating asset file for {buoy}:............................")
    create_openspace_asset(df, buoy, buoy)
    print(f"Creating keyframe asset for {buoy}:............................")
    create_openspace_keyframes(df, buoy, buoy)
    # Generate .wav file:
    print(f"Sonifying data from {buoy}:............................")


Creating asset file for rafos1060:............................
Asset created! Ensure rafos1060_keyframes.asset exists in the same folder.
Creating keyframe asset for rafos1060:............................
Asset file created: rafos1060_keyframes.asset
Sonifying data from rafos1060:............................
Creating asset file for rafos1134:............................
Asset created! Ensure rafos1134_keyframes.asset exists in the same folder.
Creating keyframe asset for rafos1134:............................
Asset file created: rafos1134_keyframes.asset
Sonifying data from rafos1134:............................
Creating asset file for rafos1232:............................
Asset created! Ensure rafos1232_keyframes.asset exists in the same folder.
Creating keyframe asset for rafos1232:............................
Asset file created: rafos1232_keyframes.asset
Sonifying data from rafos1232:............................
Creating asset file for rafos1455:............................
Asset c